# CFPB v05.2 — Nemotron Super 探索性生成 v02

舊 Nano endpoint 回傳 HTTP 410：2026-09-01 退役。**不要再執行舊 v01 notebook。**
本版使用 `nvidia/nemotron-3-super-120b-a12b`，全新輸出目錄，不覆寫舊 run。
同一份 seed、既有生成 prompt v02、Excel/HTML 阅读流程；不是 judge notebook。

官方 sampling：temperature=1.0、top_p=0.95；明確關閉 thinking。
本實驗設定 max_tokens=4096、並行 2、每請求 timeout=120 秒；不是原 Nano 的同模型比較。
官方依據：[Model card](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b/modelcard)。
官方網站列出模型不等於實際 endpoint 保證可用，因此先執行第 5 步。

**步驟 5：無申訴資料的 API/結構化輸出測試 → 步驟 6：才提交 20 筆脫敏 seed。**
兩個 API 開關預設 False。測試和 SDK health checks/retries 也會產生 API 請求。
若出現 401/410 等錯誤，畫面與 `failure.json` 會保留底層 status_code。
不自動換模型、不跳過 health check、不自動刪失敗 marker。

原 11 個資料依賴不變；只需同步本 notebook。Colab **CPU** 即可，模型在 NVIDIA 雲端。
`privacy_verified=false`、`benchmark_eligible=false`；本輪僅探索，沒有 LLM judge 或正式發布。


## 0. 連接 Drive／建立新 Super run；第一次 RUN_ID_OVERRIDE 留空


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, sys

RUN_ID_OVERRIDE = ""  # Resume: copy the complete run_... name printed below.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive", timeout_ms=180000)
        except Exception as exc:
            raise RuntimeError("Drive authentication failed before any model call. Reconnect the VS Code Colab runtime and complete browser sign-in; do not change the API key to fix Drive.") from exc
    PROJECT_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval project in VS Code")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)
os.environ["FINDISPUTEEVAL_PROJECT_ROOT"] = str(PROJECT_ROOT)
if RUN_ID_OVERRIDE:
    _FDE_SUPER_EXPLORATION_RUN_ID = RUN_ID_OVERRIDE
elif "_FDE_SUPER_EXPLORATION_RUN_ID" not in globals():
    _FDE_SUPER_EXPLORATION_RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S%fZ")
RUN_ID = _FDE_SUPER_EXPLORATION_RUN_ID
if not re.fullmatch(r"run_[A-Za-z0-9_-]+", RUN_ID):
    raise ValueError("RUN_ID must be a simple run_... directory name")
RUN_ROOT = PROJECT_ROOT / "outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override/exploratory_nemotron_super_v02" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({"project_root": str(PROJECT_ROOT), "run_id": RUN_ID, "run_root": str(RUN_ROOT)})


## 1. 安裝既有 library


In [ ]:
import importlib.metadata, subprocess
PACKAGES = ["data-designer==0.7.0", "pydantic>=2.10,<3", "pandas>=2.2,<3",
            "pyarrow>=18,<23", "openpyxl>=3.1,<4"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
VERSIONS = {name: importlib.metadata.version(name) for name in
            ["data-designer", "pydantic", "pandas", "pyarrow", "openpyxl"]}
print(VERSIONS)


## 2. 載入 hash-bound snapshot 與來源驗證


In [ ]:
import base64, gzip, importlib
INPUT_INVENTORY = json.loads('{"dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/analyzer_config_v02.json": "ee8c24e33bfd1dda8ac09f790c142842750dd7748050b03365c1d1cac93f16e5", "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/challenge_results_v02.csv": "1597055cfd83f2288d327334e9fba59dfe2e1975ac508130b82bca88dd721274", "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/negative_control_review_v02.csv": "ea999a0cf52a962a473f88bde686922dcc7c4ec24136ba8f8007c631dfd75f5a", "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/pipeline_privacy_disposition_v01.json": "ad0fd3af64f916bf0078542ccadd9c044b8e5585889dd08d438e7398b3b64bd0", "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/pre_pipeline_override_v01/backup_manifest.json": "75c1a479dc161abb9cb86f65937327903cfb92ead6723eb277df3afd1eb5e83a", "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/presidio_spacy_findings_review_v02.csv": "ca21bfbc14a302490f032b897dd24f3127c90b262cf98717ab4d7c9d64d704a1", "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02/scan_inventory_v02.json": "03c29b97c0e303990e4f061808958b4ac74bee4ace1171c5693c9043a4d1e13d", "dataset/curated/seed_pools/cfpb_dispute/seed_v052/cfpb_seed_v052.parquet": "2d2cebe5d3102a9f2d34ed4b40dc324d6de6521a0d8fd4541b940ac3ca082fc4", "dataset/curated/seed_pools/cfpb_dispute/seed_v052/cfpb_seed_v052_generation_input.jsonl": "7bace2f8ba79523eacdf6a93ae0b49450154c3df76bf1400a187009adf5397cd", "dataset/curated/seed_pools/cfpb_dispute/seed_v052/seed_v052_manifest.json": "2874b4fbae4c7f02073c7db013968ab050fa5a221b378e1a7f7ce70177d21b01", "outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override/run_20260722T135306Z/prepared_inputs/nemo_seed_v052_pipeline_smoke_20.jsonl": "b2b9b26f23eb49a7086c2fd8e2a6d6d986fb50bba97204d639137b402b18fe5c"}')
EMBEDDED = json.loads('{"cfpb_v052_nemotron_exploration_v01.py": {"data": "H4sIAAAAAAAC/61ce3McRZL/X5+itok4zcBo9PCDZezxHWHsxRFgHGDu4tAqJnpmeqRGPd1Dd49koVOEDBjLZ4PNwwaDeZjlwGBsHscuXmPgu+x6RtJffIX7ZVZVd3XP6MHtEkCou6uysvJdmVljWdZTrZbn+o7ohE7HDu3YDfyScNvtbmzXPUeEXV/Ug2B+3nE6rj9bErbfFKGz4DqLYy03jGLhnOoEYRyVR0aOB+L5bnPWEUEIcMGC23RC0bA9LxKeu+AI1xfxnBuJdtDsek5ZHDnlRjGAioWJKUxwF+zGEsEfaTqxE7Zdnz43RGPOacxHwg6BjdONnOYBYYuG59i+GDJQYIHjT50UdqPhdGLbbzjlEcuyRlph0Ba1Wqsbd0OnVsMWCW0s5wcx7zoaGVHv5uK2p/9+Pgp8ObcReJ7T4JF68uGg6wMB+b1px07sth39UT+XBP3/xcB35LiOHc95bl0PO4HHEfkl6Dh+Z+mUpz/9RxDOE+1LwgvsZm1RPWYHl6N4yXMSlB713Fm/7fhxSRwN6P+ADxT9o67n5SZ2Y9dL5s06cQ077Lb9mufEyaaSwbR4NOc4cRn7shdsz20y1fT8x/D235O3IyNxuFQZEfiHwZT1MCllTq3R6tRrkeM0awsT+6ZqHbfjkBTWonYw79RIHuyIBxsweM4Ww4F7u51iw/QKnUYQNqOSiObsqX37ay3Xcwxwag8Kl63Q0HjbYYTXcdhtkPw0SyKZHgaLI84pEjZxjAcfCcMglJv/x3f9z930P3PPIyeeeuLY4f8UVdF0G3EhByfwvaXqybAL8WdTEEEqbC95w7peW3BCt+U6zepR24ucEuOZ/lN3/MZc2w7nawA868IaqXGi0617boMlLf8tB6NtL9Uc0lEQAuvXSHgjJ9aAmjAcoVvvMiRYKrce7gZqKwjbANZxvSCuwb4Fi8kepAWsgdovOr58Vxx5+sjRI08fOX74SO3wU088++TxZ0C0aYsFwW1aJWGBRE1Qmf6MuvWa8ehGUdfR79VDDhv1jzUbwhw1YU9rDpleWD6aN+v4DjaFpZqu7QWz3dzbqNsGjZe2hpqM1FyDwXQiApKxvrWWZ89GNbKX1szIY8eePHL8mWNPqa22bIgQKNYCXp4bL9Hsrh91OyRoAN2AqQKbGKjCsvYCJqih4Nl8DZa/1fV8J4oA/8hjx04axDSWewjLSf9UazoNljtFYbCznSENDaPvoMHsrBPFPHQ4GRJySNCQp3nXb2og/MI1H5u1btxIn6FFdtwlxFOcB4XiIWFua+Tw408dO3yEhi4/+ODyvLNUwdY6dsS0b9muJ6nYcMLYdkF0EkuBYeRnU4KsZHY0QBqC2Y3I1RM0HyIZ1TRdJPz0q7HWMKCaKARyrtu2eb7ntTE6TwfeicPCahLNmlkZGYH3bwlQD3K2WChKOxo6sEB+4lLL9El7VfixRrHsRgFrZVwoKhCLoRuTHWo4BXK5Ffa0JaFErSKg+eK/RH0JwqxWQZjwKCmzIBGBSNseFoZ+ODCnvgNjhceOZzccRA0Ie1Tw0nRbLSfEBEQosUuSXqZ4gwB27CUyyuCgWrUM0QuaTsHqxq2x31tF4bYQr7h+xIFKQY0qEW5F4cB26IkKXDxXJkcCQO35phsW5EOkzCqjVAvm+bEoHVBLzuJPkSan+SV04DWYCoWi+F1V41zJ6EFou8AF/r3rsHcrtKwkeNO7VoSIDghoKiI031mk6LEilmmdFauYQJTc5MdFN56TeFCkUbBO1UEUOME5hIGekyIhn8vM04JCUTM6shcctjsZPi8Qsmq/OVEocVRXbnbbnajA40A7WCN4aDtquK425RGMUw0KpekL4Qa1q1NFaKr1R9/SCLAbW6q5/gI+B+FSAcbmeYSKcJNBrPFJvlbYXyrMSGVDx4PTWWAGdjDNaZIGJ+PLwL2dYR3tATJlriLGEzAmh2GqJXXdiCMBE0rKV8SGzvEgPkruQ7P3mSW/AZAvdN2Q8el0sZNlvYbJTSxjxBpMYBYkvZkdJekYARfPPP7oGICIthtBixtzudUkpTnCh7cAxe1ZpxDaizWT5yrOahov1YbJ10KLMQGEMwOlQmZOsZT9qBcoJszy7LrjAVCwGBGXCgVLA4Ah41UAo2BhosXLFU2Va0bkDTG3jHC7kAQARSkGZHh8hjxjEtf2lwrER8NOQCaVjcA8+kZW37D+WIg/eVApRDsFemae0At62JEny7zPFehdRByJSNEBsNntcODl8FbFscciK7Ezy2Ea0khfFPJ+7MUVWnqLz0yylRSfAVysp+3FcU1joVmfiMkBClNjV5lqL+b4OXLCBYpX/UDY3TjAQJeEWYUy5NmKpltJhUOJmY7XYb6G6jLe53QbElzLaTgB9JtBmwP+6tTE1P6JRyYfUaTf1mQMACwmMiyVfUD3LRXXjje6HKuN89hOEHjROIf6CHQB0RlPDh/KQalD9wDAwgBE46A8njnH1Oxu043Hre0DJzh5EI2p8PDknpOTe/ftndrz3LiOKV+wxxHcefJIgjPIuOJQ4DWHIRd0Y2wnGk85Op4eO8a3OmaR7ITw7dviauA5NXVycs++PRP7n0sEsMacicZ9px1sfZKbmiiTh/HSTdTYYORMD94n2kOaqcexqk5NaA3eQnP06BU1fDsdOqIdi3MKntpbIuhd332hS3kflxQb4Y0XdChxINVBo86EBuJa5EnYcsSQ1Izsdgf5oqqeMk7R5K6INCIPar7bQgyeAZCbIbVCj+TZlqZf8nZIpOO3AoBln0/0jwrJYA5+YudUXODADFauqkOzjIMrEAycK1KFBidAdeMFcUuOkvhbM+Aa+0U51PSRklbFASlMQDBTanJYDckFr8vxOIEMumEDBNkCMonUTtbdOpFYUxsOVcdtMkPACTowW/GfYtBMzCg3l1CZ04HkodR7xCohBaa7QEKZ7cS0M3dlDIsYUltx4ca8RmMuCMyw0ghAJJlKiQiUNMcJdFnbckOEokIGN6lb9KGamlhIX86MpLZGDlbSmz0wshGTxheeYTt4FMhD/+IcAP6ud5Kdn6KSVYEsALL0QeQynoyDtvGmPmmzaw5emJgcBk8yFiPDqvyzpAhei9wXHfi1rJ8z/s7CYSlu0tKm8agmNpCtTt5CmoqijKSMsXZlIWVs8Y+aRyWeZGp3DHbkiuJfxE42eztsTgy1yIK8l2d3rDQWVREjL5pR0kISYgatFpw4RRJ26Np+nKTc6OwZcexIZ5thliiBkc/WpVP5nLTt3MEc3i5mS8qRWbZURJy+9e0w5EMBmUUn7CRDdjQ4z/qR3XLy9obO1yGYnosIh5oUFR3CMgUeEGDrCX7ai4V8QKgXyZxBkBU4rrIIOqlGqVYE1C7JnN2C7SRbKh49cUxQ1r7dwcKY4XOaIBaU83GaaXYhcDmzqY6Dpoemc4tSXDUq6y3NqUM8pvqcdZqZObtynPK0VTAQUzAojRXPWTPFsqJlIeNt9YT0K8uLPEyoBEhEImF6Pjprcf5CL6Fd5E5ygfOFcoSE1DhOPHPJ6WKbY67icHbNxLjtdvETWWnMuOIwRQwZdMeno5+B0OBZOJW7fK6FzzY8CeaZU/hkaiNOvhYKOckZp9AfGcIIlSQnHNcHAeWvxmj7sITlWS+oF6wHEz9WTIxjukjGKpGlTj8x4SZ3os+j7bo72w26EVOjTjxxEB3gJEwWm9NHkHPsEnvw6ZAK3dmaRunq0xMzw6lFfiej0iWxi1lM48y4kcQUG8Q1Qgic5In4Ui2LQ3RQ0uJpFBiR50ylZcElYhgWgo/oOOiiFgSTBZPCRCqLEzqA0hFdSrW6A+9hxFJlbfxgNdT5cluRSMyInjAYDiZfhgWEw7b2LJvheXK0YLTOKUYyHjwAEre6nISwYQnmYJqIAlRaztrt41Rl1VZ6gJXJ8d1e3NZQ76xZOjNFlIJJSuxUWX+oxUFh0IwVy3ZUo4DrlDJ4aepyN9a7JJYTGkoLWjGSh9rgVQbMYkkMGKbKcGO2Ukyox+n6fB7sNybatkjX5eanwldHRoTO+8vw50b8VOEQwMyRIaeUpkR2O0sGZyOpnnM2biYJpzjKGoynaHY1RW+aPhvLzBhmwAaXsPka/sUUs1pKEVnJQLcoxAOQVrxBiEHZKSobylgPRTEPUVY5gRvUn6fDTK4am8Z4SV3PcL2oWUQgOe0Q0+U4hDpgRGQmwJQeG6lFPRGJUCieUmjPK5hDSjKDzURrE8X0pFwMphEDEkiXI2Fefh7iTNlFd6UslrGXQptRGw0DzxktidF/HYWOgAAODMbKH/1l9VkVQWjEaHFlp0yTUaUVbkli6Pgo41DEZWxw0qBX9pybQ96oFgwh+vDaQVIryPkWklMu5dHuSR4YHCd0LTDQzOIOFAinK/tmVnLwQC2SscJg8bcqBdXwOcYYFTvPbFXuHKwaV/UfO09RJeUqbTGRvmjJh+GmQnHDJkHWZWfe9s4wM8XnqsGRZIF8eXp6Zihrtlxrq2q2uZhS8S1h6DqtZZmMNKu6K3mBmM6XiElbdG00NzYq2x36VJCPGdenRujip+oVqnF5KyoM2OnFuj5r66GqNMaenrs3VPMC21nq/MGExfq09TSja0m7p9OaIFpBtgfJBdMjO32hUYiJOORTdNg2488rJHsQsi8pgjux/Vk6f5r7nubukxfdTkFBpvKLRUsvcCsYIgJZSSVpXJAKtqDPjMWB0sv0ZAUmPVPXQ+xB9d0GG/KoIFsZ0mKdIqgKcOXXpNiSDNohVSGnrQzO+y10om0kBZIssXbrIxUeRm2yxeZzoJYnvSKDZfXDQAMuIpWVYr6IpW0ntIL3SVPw98y2Nm/nUtXTmj1CWz69dy4htqZHFVqjM2khsW0jGZiR/CQ41A1UlCgDIVP2bnFYHi4lwxSwmEAfODsgGkgCdrhbKqGHZm9iOVVa3RWoQkjWv8V6GZzTxd/FqBwjpcB+V2mrfq/shyauwWd2Tm5iRyp57nHuDSqkrBYxDtpzSOyZenj/wyYLd8k4JN0ajifwnwdSt11K/TbAKGrvTKL9CNEpYiIEPX4jH7eku5k2cBqCyYwmVLkVOs6LSEHaPkdG1tGplDCoE1IwTME2CX2VXjZxPvEj7g3VhIItKmn4aVCRmJ5JY9sAoDoqUzjTA32WaPrwijPlRbfJWZxsavrhCSK/Wm559x1eK2zyMqD2TmVBDTaA7b5JSoIXU3tTbhg2BPumU5/UAzhU2m+V6da2T+mHnHiwGFQJgj47Dw9PaVxZtWxQAbwy4Ih1oiEZmAhpZajT/k2CSZ0tdBx+/OSTT+ieICmjSOApKTVzNOb2uHG2Fi91WC8ji3SeTr3diDLPlEvjxCjDRH+RxIJ6mrqeXR4gVtnWjb6AljT9FhZDu8N5OdWyAmDcyFS14qCTQ4zBQOSJ8EaXMEK1wONmttbs4cALwqp19OjRqcOHLUOAMv13LAzWkYeP7D+yP6ugtERhkpquvGK5FTCy1JZcqCMTrlBsJGvgn+2m7wbTyaN7jzz8++zpRlsD2U6Xi/IXqHsw07tcIBZVLQpcCLJkwGR11BpF289oaVQeYhQ0aQfpg4XTCbeC1uqe7Zs9WOla5WguWGQpe1KeQrD4QPodwxwa8gz1dbOoEOsyQ8heNZvczFpLW7ELzYWBBTGMzlvDrc7KVGWrL8ukQ9pZPTS5YlQdXHkCh4MtTJWEOQ5kmMraP+izafzcmfKc487OkRhM7Z3gkbNd2BrpwhoIr2OqbToUz/+BPhjr8i0FLD2d5kD6r3+y/v0n/dXPBVf3fr33/v07r/Xf/RrBlo0LAn9fPX3/7t3eD/8jZiFueNKlMLo8EJI9Ef21K2mf8d9XXzKKX9b66W82zn3X/+qV+z8R0Ps/v9q/e2nj1rn+W+/9eu8CVupd/Pr+L7f7b/8VC2/++HIyeP3tb4VuiRS96zd7X9/Mge6dWdu4vSoSew54vVdfW792Pjl2YkJv7dXN6x/SdtpObBOz5bJ4s/7ny71vMP6j9ffv5EDnW20rAuN7lz67/8sH/Qun+6+v9b/67/Xzq/0z50GQ9bVbG7e+wSq9D88ztc7f//EvG2tn7//4bv+rn+WwX+99ZMIf0rZbEdSVCh72v3uzf+3c+tV76xd/3jz7Wu/Tzzeuvt///Hz/7V/Wb13BAli7t/YF/XHul42bP4P6699/s/71am4P+R5gvYeNs1+uv/IXmn7tXO/rzzb+98v7d9/snb8MumysXtj84EMsuvHp2d7tT/rXXuvfuN7/4Jcc9rlOYg148/QXvUtr93+6tv7S3c2zb2x8+JqkDOQtRqQo1t97JZE1DOyvXQL5N774Nk8bbpWt9l7/qH/jT5CO9bdv9K/ckzMhI9nu2qoUH7zXPbbVzWurRLbT7/FL1WxLbzduvL5x80qOSnKS6F38cuP6jfVP7/avfdX/6nrvrdOgy+aVbyFgvbVP+1d+UGpx7eWNz/+0efVTqStjdMRj8ddKMQvdy63QP7dKlH7//c0PPuldemPz5Z8gzczscapFjSc4kgJ88KFQGG3cPE9Sf/ni5vUL6+fO9m//eXzjyy/7776eA59pGxa9X85sfrzWv3FVcPMw44buYYB+9JjorX2w8cVlbGMDOLz6Zu/MGTkqr1dAz2jIFhBl8ezJw+LYM0+J/tWXNq+8RSr08/neZy8J6vEZm3hkbPKRk5NTlYkJ/Puc1MONG7c0lGZeMgeTBKL38hlIUf8dCN25jc8u9s6dJXG5eGnz1TeB/PoXd9ev/iSJsfn+qyRmd167/+OfwLONLz/sv3KR5Onje7l1Ni//0L9ytnfrHQgJtrzxlzO9s3dVTNBbe2fz5rsADTpDwtJcLWwEyF+m60sCtqR3+0L/8ve9M5/3bv81T6dXvuvdevf+nc96ly70b/2PBvzzBRCAdOiHN/rXVvFIG3nj3Y3PTnOwgCV7t1+WstR7417v4k3oGCwHOAUjCCsn4aRrGeUvNvNJwE62XIXl8sNgoGw9aqVR8eTkvnzVm+dlzqOIHP+BqIhs7DA3VFPnQGNYsg0jM6RG5ZuGizPGNIbIeR7263NuE1G29f9t7QamVD6QJ0t1rK13XeowkALRQT1lN5n7EvV4UINHJkGUpsi3qAUMzeIrMlCmgcsYW9UkdLftLgsTang+rrY6iP8aS5gk7yJREUQl1GvyHg5dF8RnTmWt5EouatdccdFbVGf4pN5iae5XRMJhi1tBKppCKtGRZGGqwgBsZF/VGmh7O+VFp3TXm5mGWEzu+w0kIZSQ4J4l36c5+Ltm0OCjBCn7oYOs8gg7Z6ujL86NPQ4KjB46yBKNVEhIF55GuTqPt5wZOPSHBC/F3oPj8kP+Aox1kO8XHqoHzaVliuArk/s7p0S0hFNLe6zrHsCZbozVtDK5b2KicwovwlnXr0w5bW6+PdBBDIoQpzLptPOpewsi74bL1IXk2UuVWTRnHqD/jVFlE3UtZ0yl/iqTrVDgvwOzdocgr0AwlhfncMwciyDmTgXPY6TlB/IrUDqshaicv1aQyFhE9dA5wDtxffztxgfqUBQZilUeaO1p7W3t3xZpKk4iglyug0VOOAYTUpkiktBZRDxQr9cVCeQXAPi3toOIppBS6vdEqOJydvd1L2jMr6wcHJcEH2TE3OShqYkxMn+zee6Jv62+Lf26IL9+cBxjD3YOnVCuvevr3p0yFZ2eeOJJeU2tLPJ7w/0XmdNVGWgFH3d1uSFEXxvi3nc29mXxDHck0tnHmZXJKr6qSx0belXBtdzywfHOIWsmb8gHEk5O1KCUst2uN23B6XQS7zJe2x2Hsojy0GVch6Drx8omt6yDij/QiqlDy5hVSBKARVAXL0EZi65wkILK0xyNmlfFl+Gll0nUXorDs/4PCSyKrR1sugs4WiA6qo4SZ6FskbwwDFT2HPqDDvaBwx7gEDoKu8GsDiNKA6ztVkyAZ1bReZ3kJDGw3EC6aFfLZWCrnNHWoNWABLKB7TjItO1KnUOPZa51c5SFXC4vsVWVRq4E6Jr7lmEztXCAaGwv1TfjWtAwX2CEU5QIS+qXDFLVDht0A5xssroKripEQ+RbpVn5K94ajVWckt/6LmWxuLXfip3GnE+RTE2W2uEuEX4Mdgs8+KDhvbg1gz2YeXgvJVUn7amGDTKCd3ytUf9NbQjq5Ma7bWR5Aq9g7nT7fRqXRgbXy06UhKxJ8lvywkVBPtFOOJsrfwHAaeqNTBjQqEiIV6m9Qx4e5xgyVZJNTf6xA9enplWyg7jvS6kg2edNaM/Sn/IaQtlayZSkkov66n4a6wKaZRVhEz+fi7d0GSPTdZcMVtHh1tFKceemvDSmy5cn9HNxZOuSRhKUJO+nZSg0o6HLI1O2q2IrK6/y9CXq7HYbTmSk6AZvvOl+I2X75Y9Q6AfVG6jgDGZ5h14547QZl+JXKrIqli0T5VYdKNKifpVeW83lshFbuL6R1Uuu0e3movJvumA89KKYJkyZ7rZ3CpmeAo1KcRe38pKOMhVjpF2I4+k5bVviIdby6fcB9LVdamtML+oadNXbKasLtgXrOdrrQ3wqz93FIKB051e2NheKSZUXu8x/K8cooFFDPnjSVBdcJ3YqTJkoiTbS88gQyhu/sleXMwq0HwhDu7O1rCRcRUm/qtM1VtoiR2ySgsSXutPoIxWSHYtoKuOi2Bpx7bCFgEykHBJxwCmbXSH6uxyiWp4GS0OJeO2EIngzlqDJgpRiKxRYoRpSiIdJUileYpLbfr7ypxmUdEOgkq8sEP9wBR0+EwM39FyZmLv0NDfczWUzVc2sT5TvMp5zEJYYy43Ow05iauoMy9+kXx4SJ9En5fGSqGMYLw1vqgwz8X06Z2FYOOmv4rDwlmWSsnOUeM/d6l8Z2IpsUDeyOFYOz/khSA0Re4novNQb/qWD4nCe5AqT/MMCaof5muX29BgYPdAZNSTXoIMIXB9B8Umex+j4dYD6ceUvIeEMHrqNiK+idoJO15OjaBWYQCd3RRXa0A5oACIKGfTBERreOo0BUPqSEXaNTjJS7qcHhH5mujI5NZOPHwlomSO3qNtqoRfVUgFESQFSE+QCnG9AryLu8sUhUE/y4Otvf4Q8uEwPchsXimVW74fvel9/IZYzIr8yniknrVAe8eczPE5hPmoqEHxINl2pOUCrUJKbl0Tae/3P327cWKMlkelcu4RkZVp2Wf9+dfPCK1hJH2SRtly//Y5MygM8cspU5+DyEBVarpyX0HhlLJUGMCT+JR3os/RIYm+jtzODEYyiZnpMHUPwgbGIPoiDBSOVKJca2rvG3fUSE1l07X3z3vor15P9I0eOzL92mHpNRIScrmQCPvCAoNzunTtIKKNMg0m9T7/tX35Hpvh/vbcmK0wyebx+7+7m1R9EbJ8K/KC99Ou9c5LTQ09vmT45SaNBgzAzvCdPaNye1gyUlbXej3dRhpEcGb4ppmS0It1nhkFDDcTMDPmYaUU4mURHNU7WS2RKg6v7JAcz25Dx/p3z9++sInf+6733qNJy+8Lm6lW6N7Z+61WZR4eAbp6+0+eSDuSVpHPyb6tvTYnNj8/IzSkNuvLX3r2LsrSHAhfKWf23vkfpDID6l+8M1QRlkPK/otQiWmJLQ344SX+iKo389SLBv16EV9JQ9T++3r97pf/ezd7pa8bejSPyELPRbpLRSE/GilLp72Nk+hOJKyP/B5sGxLpyTgAA", "sha256": "f1d8eb41b2fa28b0dd576de2b3415e744f0b76800094dea75780bafdff95120f"}, "cfpb_v052_pipeline_smoke_v02_common.py": {"data": "H4sIAAAAAAAC/+1ZW2/cxhV+318xJfpAJitCNpIg3WKLOLYEJIgviNy+qCoxSw53x+LNnKGktar/3u/MDK/LlRyh6FMFw7vknDn38805s57nXex4LZIly+Sm5vX+JJW10iwuC13zWCuWljV7ff7hZ3YhRMJuTr8PXzKVl9eC3fBMJlxj/eb0ZbhYfNwJlpdJkwmWCLATNdci24NzLsEoEVrUuSyk0jJmWtxplnMd72SxZbpkO7ndnVS1iKWSZbGoannD4z3jRcJkocW2lnrP4p2Ir1XIoEzOC+KTyiIBB8VgBasFGCgB8oRx1SooksWHfWLJwb+sE8VUU1WZBNkGItiuATes3Uhxy2APZ0pU3KkvwKYhLuxTk2xFuPA8b7FI6zJnUZQ2uqlFFDGZV2WtoW5Raq5hgVos3LsdVzu4o338pGCe2V5xTQvt3g94tAt6X5FX3PtXxX7JfpPwHs86pkWTV3sysqjaVxV8hRf4VyVOQGu2o/iZK/G2RHCW7HVZpHL7RsZ6yc6lyJABKX1EXVQXi8XF2dtX7z7+8jr6/ezVxft30ev3b84u2JqB+RdRKKH9BcPfvfmf/jze6F1Zyy/GBZGCK0QU73ixFYm37MnijMs8gpmzqykSr+GZ2d6oOQoKOkKpIwRbSS1vRCQLZGpu5W4bmFDEYrxFNWkqY4n0iLZ12Zi8GVJkfCOyfilCDaQZHDSiEVsoVtZRjXTV0I0MGRI0BaUWnC2SqKpRDVCyKsFm/whVLJIGwZ3Ve0iqYlHwWpYRaonLzJE9LALECpooxS50DY1NjP0u2sHK0CFv6RXVqMiY3nGNlP8kqMhzqVQlsgxJjvRvCnFX4T2eUBKJgEY2OVRocp+YGSbGRXKLjOjTyUdh13ztIRobmXi9ameO1QXy1B/o6bRTZVPHYtUm+qU3iBHztqIwcJLQgwkUfUGtCe/KbCc8WTGlayhj8tkH1ESZKLZ6t34RWBGa1yACnLB/s3dlITraRKS8yfSaXi7ZVqxP7Q5RJF9N34XAQdOvQIscyTZjK7z4Htws7MDhxpknBl6APA7ZEgeFBoE56zzA6vK2j4MCKkcyecr0WnDAjloBjJW+BO3V1JaIqq6s92uicNa7iLldwwB+zfbWgEFMeRyLSlPobObZbwS7LozGB9G1JL93u4yfTOCzfET3tN0orrzSVLBUWziVChGpHX/5/Q/jnYBiCCvWtfevy9OTv/CT9Or+h+8e/uwFg2wveC7sti4d6MM5mKxALBodGxobnp8moOp7LhKO8U8mZ3IB1Eyc09Lu3IosLaosEcqPM7WciWPATv7WP6064GiK66K8LaCjMuDhE1y77djDZrE96LbLtOXQszR2cgkI+QeORHFW1zAo9f7uJHWZa6Uwo/aK3Ts+D17PvRY4Nos5zfo6eo/+IxO/l7fzBfSWU+sgTkxLwDdoOniCrJCxOQKoSP7KKmEAhB5sgwDAYKbkiPUziohi7I6lQX46MX0yA6aoQLoFW00OU6OZuqCtj9UGMZtwQc8VPauqY7QJNr2ezaI/I3uIsK5zWnZgdYyAFzzbK6kigvDJmvNhPYjIeKUvMrt0pNLmXEVOnXXA/+vxaD22boBjRK0jVwUxgDVDSw/qLDVGExauhgrTSjgqGvan9aBGnrTEQgCVL7s3zFyhPjCJFl7LLGtL3Bt5isrc0B/UHJ223eog0ebe2zR7vo4Zx6zSThU4iG7QvFFz10Pcuemjz2UmZjCOpgN7khiIkl9EtNlriiA1JG2R9r3KM441wsU/3N9Ypd/yQqZCzXU3uVuKbkQ9wbmb0xcWzOiLtbKpq1IN+z6Z540mQI8qWbkTm+bNDkoMB3/Q2T+5wdAGgzYzqpuihZfp67HfYxSCHh7r9DJFxNqa72N41a2hOJpCHwTKHSMaEyKGmycD2kEjXE/1ZwMckQDfKkkToyk9EFnfJ3ILz4OZGzpDu8m3HG+l3pm8CkvUDXqRjRfQvIgBK8kGtUsd5yYr42toxigufsbzTcJXjhIVwhP/xenL79g3jD6CJdt4XjAuFqtL2FSEnb7h156jBm3c+k7c2W9+4AylCTlSPBU+jd7wASZgYyY+rQjUOMq5AKggvJZqCX6xHqjgpNzDN/612AerAV/YlAfGTKwsycScTDWMQnpSfvBwXFJRhRi165rvD+VdzkkZC9AlZY4fXB2X4BPFkukGGBs8R8jVo9qb81nGh4x7D7iUmeVQJeFHmSNkPK+O8lClHcj9UdDNootzVvIkcjcy04w2tUURpWN0SZG/coepGePvkOImke1TmJXA8V7llgZIE+Ii53MjtPeYEwGE1ocE43AhDKQUj9xeo1yAuEWkkY8LDsx1a8+p7vWBdHLB4d4Lib0ZU5Es9N176FWYVCJQqqRjbO01Oj35cb4qp6qH5D/lE+BZ5ekbybYbSR96E8KBsmqzbeYE+6WoGtzFNcCNjWC/ovn4jY7DD9Z2NAyk54PXVuctbuJEZIwbBG3pDpO5uE26A2M2PAsfhvk1LnJ8+6DWH2vKLnEHHlF5bR7ngesWbp36DIktbsngtffP4jiuuQgbbUeuddBmzPONfxPcsSl/nCUkV9GVH1exlOtznimoTI1TBCixJgTsW2Z0cB6Decp6LDKFP0G1SbM4W3Gm9z7MYMK2pwDAPFHDZE54kPmH7FOkkg7MZStSX6qC25U56OnvW6iXhQKWsk2zxaAHM5cyRxgYpyRgMUhloh8xGGhq6Q89Mc1n76y9u+Iml80Gb4RBR9xmJVwNwckOilGsbkap/g01/p8bXEF2bfCKbcoyM/Hss78bYV1o0xo3CIRcDl5avji76C50bTZdC1FF7exV8DbFJul+ouR21MXNSYWoe3uMmcw3cEWWGj2Ow9kwZPTqcn6UujIYPE7tCVjR7rAqq/lhzOwjgBx078GB7Plx7Q8Jn2XxhHTAxLp3ZWhvgNpJ0LfcR0PHQUaM1QGPIwPUmAvo3AQxD1MzAP4GXQLdewh3sdHfZ9wP2I3mPbC9HKyRN/E4LBIiGZZCO01Gn9y95sGR3Wfg9ArUpX+3c/UY7bGkPegWBnnacsbeKb+viFy7e+j4Xtev9343cHfq9IGYyhhGo5N1OaUiZ7TvhsHpdrgIIVh25pjGpJ/t/lvNU/tj0x79LxLIEZhfnT4vplgvSaPPoWslztvJJQhxw8IRDh7id6yIUi34X7dQqsn9F1/RNs26BcB92E/istCfInswOnjMbfF/AGZVvsh0HQAA", "sha256": "a35e8fdfcaa7e486f7b28d67d587f4cca8ff76fa5c52eb6ba8806f44804d708e"}, "generate_multi_turn_dialogues_v052_pipeline_smoke_v02.py": {"data": "H4sIAAAAAAAC/7VaW2/cuBV+169g9dLxQqN43SucTgHHHm/dOrZhOwsEQaBwJM4Ma0nUipKd2cD/vd85pG62190WbR6SkUQenut3LkwYhj+oUtWyUeL49OqduFEqE/f7f4gPhCo3ulSq1uVmbsp8J2xh7pTItMzNplVWPOhmKy7UexMHwe1WicJkKhcbT8/2K0VqykaVjSAqsRBXtcnatBGyzIS2FgtyuVK5FbUqpC4DSzxUtblXpSxTxetkjX+bRqZb921Ty6KQjU5lDs5WO9GAgcw8lLaplSzEvcx1JhtTx8F1W1reXyt81GkDCo0R3+/P5wf7eJmaOrNiBbYl5CE6LKiT+cHUd7aSqYqDMAyDYF2bQiTJum3aWiWJ0EVlahKlNA3YMaUNAv/OWLda1ptK1lZ1a4/qTVtAHVf0snZrKhyd61W35AqP7kOzq6D/7v25bqDb3LNR7TJZQgXd13fSwhiwQSROtcqznhMoQiaZsnoD08SwxlpvhISBMkdo+l3DWPUaInd0T/D5xH+N+Mmq5hhqJnmvlW3zBlIHydX15d+Xx7fJ9eXlbbK8+FEsoINYlfe6NmW8Uc0sPD27ODm7ufpwu1z+eHQ+2RHuBeNHbJ4FAn9IF7NntPdiWNPk92q2x6v0Wjxbwx/gV8rTSNY6h81GW2MYBpawn373OdgLbt5f/mPZHT7h5Y0ITdtUbWPfePeG5G/YTRJykzfpulol5LcJYucgqXSlcgRPAh+ua52pMDhZnh59OL9NbpbLk+Tq6PZvOGN0IE6oakXsZIku+aRSFeYlmu7Yg/34nxZnD5SPrm/PTo/A8ovUa/kA9w3SXFor3itr5UbNeo/ZO2Rl1SZXh52XfQpb+GcYiRBbtG3ga+Fn0GXfmsVxHAn4TFrrirSxCG8qJe9UzURCZxUf+IcCgffKTs9OtxybO0avvNA3JPOxKaFOy8p/xno6+ngocvD7yZMdeOaF9Oc5C0c5ZC6xG8FGYjvQ6eQWhSNlQybhhLO7EmCB+EtS8JLYFoBU76ayvnLiu1qrtfC7cCadXBoBZ0FMr3f0aGpWiQRWZqqROp+cX9X6Xqa7BMijrBcZZ3/+9YcDTNM7KxgQ6oJOZO/WaQROyvmYlc40/vwgyLDfxxHiqpR5kjlg4CibEaAdctztiflf+YezE1D02m1jJBEdsPzWCto4N/U80wBlQPdOMGHhCYvKMDQh2xChyW6xH/8p3heF3IEpQHPJOP7lCwLqp1Y1cyJtv3wRA2nE8Bb7mq0sHYA0VpAN8g7+V7JJt2CcCTBvSF1HfdowDz4f1m0pitY2YishkykVk3Ocd7uZ1luwtnZ2lsVKb1rd7GBEdU8IJGSasr6xC0w3SJ1YLjMbd1pzXg63pLymCKEI1EjPL2Bhvy7W1hnEh8mz77Zdr/XXODcPqp7twe6N0KX4FsZedRT/HmnoV5nR7/BxoMa4ITVQ9keZt2pZ16aercMPpW0rSh/ItlMzEjuH4lvPwqMHCybkjNd/6+Qhtgae1Vf4uh2L5Bg4BeUL05yatswcH/2evZcpQTtwieeUJqKcTr2QqgRtRak0e5BkgUC3HrzrJfG8PtkaFuazrJsZsLYQa2zmH3oke1xvcrOahd/1ptgjAWjdYNVerlyVs8kRe+I3C/H9K4JNLBh2SOu9Wn2tIIrFv0Af1EHw65d8ui+bwgm1dTiS/y3Eg0HEt+csPg7bnCDe/pNVn/Y/A29uPt7cLt9Tjn9/RfkZEfHRtCKlMkQNWDypWoGLzSQxWNa0ZEnKFDEGrLFItih1EHl3RMeSermc3dTENoVro742ZPKWKOm1hktTevBxmhPmZJLqyljcEj+wEde6VGPJUsgVYJlB3B8GF6JsEwfHlORAMGXuujKYa1XUG3R0Adwnz4vYcVGVNS1soO4JK1CUirM1xOF9DOO5JqNxYd6MRYgCAA3SGhBq8p4pWpbmTqnKVdGOy8zhrccq0wKJLoBVyO+q4gJeTXKVT1CxcIuQvnVB6NqIHcwEdEMGZb3Rb5zeuOxWVLmisyShXT5HuZ3jISV1xMFRngtXdzmEXSmxBEJru2WO+Z3rGYbEBWXSZokimNZQHjOVr9hkHnMRH4y96Nh5EDn44EXPHcSj2cSdkIl8NSzSqSU7s4lZZ0ipM/aJt+hS2JZQo3cVeImXck152wKN5sLpGKXTt2/db/H4iA+2Xc3HH/GcTBZwS8WfXHPV7xo+0J7+Y3Ci1qq0aq7LeaYquE7nziM/UV9TVVfNYYDdQwmc9AsSv4AJjgs1EPupBS5Sx2NJsCvvPB2ygOBYpwlQYgMeHh/7oivGrncKYT04dl+iuY6OQs0XcOql8o1KUibzgb41sDWrIYUDmULVSWFMhhNpBeBNVtsaRh0HQx+ynXubaieU3mzZIaxKYWTE1gO3kq5xm4SZ145ngeo8+CQSaRcEstyJUkLgiCRScCiZZTWFDN6kutFIGY12jav0eyNRbclry7ZY4UUkKBJy/Pvh+txGgFMfaG9SWWfDqpubC/x9dUZ/V1AR8xwRkHI0S6IAsvNGF1SRZ8wBxakkNDg7wePZ1Yg9HMSmpARJayGR4cQ4RgcyU7lhE5wYX2RQ3fO0vvWA5p5clgVbGbQidw5SQ1IsbJZuqR9BSVLsOkFD4oW/A25L61CE+xfq8uk0YjFkFsMJ0smC90/YW0OEB6CBy/EjiuKBYL1ttqbWPxNvbTl+Ggo5PsimskBkkctnWAk/tOQb4HTwDjgpzUt0E4sje4fTStXiwJxQpfYaRD607vStKvvjXYhxakpzJeuRBBYJvM0pIOTKELYomFO27FFeYfM0NxZzjP79upZtNkdJqhXK20ZV5CMeWyqD5LJjmbhk1hi16KZlBrAmdYJJTpTIeKTu54FA/FFuHXVWMCyKjXwE4sJULllzgLebjU/j5CicjrdqAOgJH80WB222VMyDQ4PKlpeQr5Qqh3P5RMJFoMtQhsu3iCCx0A3EqxQii7wTZDdQX4S5VqlR1ZF6lPoZL6hmx+wm4oK+zMjtKHxoO5osmfMuU3f9EeTYNduXpe+42bTAHXQ3tJiyASJOse4LqtooImRG5VlEmxXyUK421CIQBFnfKKY5ugvojRFDsVe1jYtQAmHSIsxiNUPVNN5rnrfBb3EU4IMYXUL5jLeS3APqbdlxOI+uqOClooh8ZABk1HyETaixCj8SRNzDd+BiWZfi2I6wmYvmMfJHv9BRu0Q+6XbjwDt5N2zsvZQK6WGk6BPqW7BoSI2vDhF7b6WZiy8UqMtdtTrPEjc1m/E8xjW3kC1yM88Ew0Zp+Q13u1kWj2dmx7z1HZFRtSvIV+4BFchra30/5xfHZA03EPK9yFDCg8q5gRjUAdEI98a0deq68EXP8l7Ur7cSgE5pmyrTRm12C1C48S9v/Lv48vpkeb08iUZDh44X4D9UkrdFOWGCSZAU9MXJMu0zKMktwhcSfhhN1llHKMH8Uy0Gwrd4jI+Pbpc/XF5/nO5ANS4LS2uPiXt0YX7PFX+Y3VPnYxeffh+JP0biz59H2tj73ws4riz+76JNFtOfTtZQll8pcCkJruvWGTujJ3JnxCz/puEP5Ka+OC/CzxNq/52Wzs/f3/SR/+901V0TPFGTw4uEwEQ2i18cBE43jcJxMfr9xADc5FHRXFTNYtJYPtG7W/H001gRvmH1+vCI4ZrSaZQ6/Ej8wsNX494dhYot8bcTh9SfurfUc1LTlgwTNvfhu8hPBTl7T3YCZg6igKHp5en94Xg68v2++MtifDo9Huy/Mkl4OjzoR2Ie7IkkEu1w2xJ2yuP8mIG/iVjP51nDLHv41s2qZh2ZSHzX/ewG+68NdsIjf6addJJI6ADmV8f9nv0pz8UdTZL8uYvbukWu5kFVYu740W3qrlkg89j+swmxxeRpujH2V1tqNnWpXldPPSAY3NkVdouBln81Dcsp3cX0cRohIy9ZYIA8e3J2NF7wDEpGPMVUiudylzhw9FtmkxgbLrA4vJ6IPzlqMT52ksZxGZfwbRwNOxEP08u4w25SZ9lA04+z8QQ9STKT4i5ptIHBUPots3A+p8Q7JwMCZBnsOVghx1oi6BbPboReJcZoNmc0CwcaIS7XgJ9zmlCFr+6HRuZd9Hl2CFN6Sgf7r27vHPJXyDO5h3pdQXe6mnsPCLvichEideJuFflDhU+ngrbHWIpVZ8ILaud9QG5osvrc0jH/oFd2NpkE05u4L5D+g8HydHiKK2bhr++SX7oKZPiIK7rUqG3zfPBJQAzmX8odfcbtq9CGAGPM+l7kZBklvVG88adRTDz5MkGb6Gls9kG1T1pzx8JwSQcnfMN6MC4ScdeY+HJ/8dolUX8SX2U4FQzMkBtg/BNPNw6A2N+CwZnWfQpy1/bgoIdnjHoGhvppPG8LhvH35fnZ8cdD1wZZnhUuyAPf9o1HN/pdrNEy4f1gwRBD8HSLVuUuAQsbje6oW+RnfVRIGBoyUo+Ee34qw+LhLo0urxMqhvAfChYYSiYJeXeShM4PnasH/wIisV/vrCEAAA==", "sha256": "68288e5e97b325192e8f5f1ec5eb8d2edc6cfb9cad533565fa560dce043d8581"}, "prepare_cfpb_seed_v052_pipeline_smoke_v02.py": {"data": "H4sIAAAAAAAC/7U87XLbSHL/+RQ4XF0CrEmIku3bLO+4PsWWHOW8smLLW8lRLBRIDCWcQYALgJK1tqryGnm9PEm6e74HACn7sltbMgHM9PT09PT3jO/7FxXbJBXzEi9lDavWWZHVTbb0Xp5e/Kv3nrHUux0/j468TbZheVawUVnk9169Lj8yr07Wm5xFg8HlDTyU22rJvIrlLKmZd5PUXlJ47NMmz5ZZ422Lit1m7A4AbqrsNlnee2lWb8o6a7KyiDzv8iarBxuOTeU1N6xiq7JCgKttzWoPHtZJ7i0BfJUUSzb0blmVrTL4BI1NYDBu6mVNPYABU0ZNK7Yub0XLssquswJAFUlVJU12ywDLJas2zZB61qxIa4+mmQxStmJFzUZZMUrZprkBSGmybGAW11W5LdKsuJa9vaYk+NesYAgXENlUJaJQRQPf9weDVVWuvThebZttxeLYy9absmpg0KJsqEM9GMh31TUQombyGch5k2cL+fj3uizk77KWvyrGh1iWec6WBFCO8RKQheUdwiqvkm3epNmy4Y3TpGFNtmaypXweevj317IQQDdJgxjIZhfwyD809xukgnh/XNyrSWyAnMgGtbdJB4NBfPHu7b+fvLyM3719exmfnP/sTQH5iBW3WQUscM2awD89O3919v7iw+XJyc/Hb6wefjgwH6FzMPDgP8QkaMEOo4rVZX7LgpBaZSuv1YY+sBy4lcOIV1kO62J0jZAbi6aePZ0PwsH7k5NXcmwLlQPPB7IB2zcHyy2sPUsPatg68aYs8/pgudosYuTPbcP4e9hSRz7M5uzn45f/5czGgEsvOmAbDMOBS6Bxsk2zxpf9qm0RH42P/jj+/vDp5eGz58+Onv7tQOy++JfkYLXNc8IF/gA+MMGf3v71pG+G5baBGdQHmr8PSAzEuFVsPI5iKS5i2HZVBXvAH7w6OT3+8OYyJiqencMawyCapDCCA0MPFGcFDB0h1+ca0OmHN28I2m44uIa/bFnjYPDT8fnZ6cn7FhJ6/HVSZCtW83F1b+TPt+/PLs/enhOVjEWE7mriks6GYAKwhw6wtx8ugRAA8x3iocmPq8eWAAuX5kCIxZTToQZR8nsPZe6ibBrYgYd/8MoVyZ6nw/H4mQfjNSMuqFAKCflUg1womiQDoZA03hraeIdHAKgoi9EmT5bspsxTFL2wptAmKBc1q25B0iELjcbfj46OQpDT78q72luwvLyDEQHQAsVgUoGwBPHzexws36bQa8HFN+kIkA9/Ivx0YxC0QJmmyhZbLrNJ1C/LivcF8JEi0k9n5/Hrd28/gGg4fx1fApHO3wO5APuLN8cvT/7t7ZtXJ+/ii+PLy5N3uCYVi5blegObOaj8q1nwYnJxdha/PIb+r44vT768O3l1/PLy5NWX2fHob/H8SXg1B+FCcPugYMtk9Ot49MP8CcCb/fP//vf/jObm2/A7gDG4OLs4eXN2fhLDAJxBNEfw3ZLkMLdYq0N/8O7kPz6cAU7x6Zvj1zixz7SBfdIeNRAnyadNtWX+kL9fgE67WSfVxxjgXmeLnE1XCUgx+V1yntCPqfr6AEO9PvnPmM8f0MPBZtTJJtkCJwaziuI/PBnNn/xFPsLvqwgf5p+Phg9XCx81a3QWDtswghd//t1VGgKtrp68OJyNoqt6/iJ8gc/Bi6v089OHq/CFfE3P4gF+P3sIXmBnfwdg6jKCv0f0d2eXq8VN02zqF5ODg6v3T75cLe7u7q4i+LkffwD9w9CAPQc9BirUq2+So+d/JIURoGackAoJvdGPHjD1hOCl2TVIDyCxUN4R7yQU0l0G1gR2jcoNKwK/Wvghqsob2Bo54xDwP9hF3iIvlx+9rACrhlVBnqwXaTIRLUFZJWlwOD565n3n4T/h0Fv4fqghaFyi7QbVe0DwQjFpMEYK+f2GfeK/AElrog371AS3Sb5lE5ygPVEBw5kmtQbtvixTFvjbZjX6Fz8MO4bIyySNSba7pMxBQszQVJnBWEO0LebzSRf1aBSQMlM5ThclBZozHCrCQesA92VIJMZfSGHeCc0FfBOhhNoEoVz2uwpWoIUrcBEIxUknujSPc7CiOB6EMjcqovXHNKsCYWFML2GHD0F8Aoy4/EiP3YxyB1zrznfoFewOEZ76V0U/HwGaOEfC1mIPwUo0vYDok27XmzqAljhYjfZqUi+zbHqKsmQI1n7VxB/ZPcc79J54NLAgk7CfYmmAx4g+ACsbSbAiWXNOGiJPryeeTTaiGraUDJZzQ32KXQLswc1FhOtzaCGabaTHAv/qCl76B36oaI72FY4PilUC67AQsYV+C5qxQXLRSg+974y1q43tVSUZWJA/I7efVFVZBSv/leGMSBp4rF4mG/BBwDPx7srqY70BXCfeZ4nPg2/tSBxNkJME+b22qQyTIhCSRtsYBlvSN7JnpCljfSUiN1tw4RymHYK9Hr0Co/O0AsLOJ+4YQExjD7mDk0DiAqO1LfkMwSlk5ENJVcfVlq0n0aoEeHUNKnLi0e7QbWE+2ySPuQ6NN6xC75Aacv7ULYU/Giu/UcN0m7b1akcj7oYq6255w5Yf4xqs8W0Nrf1lXgLwGPcs2MtCBS/JiBXq+YH+rrN6nTTQ2yYC7KiJZ1KU2BzeglAneRpa+xk+iPfIp5KsEe6POtBNgbO7QHq/m/LOBl7QVKO2k8cvxHJZjLEpwdu/VyCAuzU0yd8wBDgzrUnCSv2yzdDI5Y5GvMqT6xr28WweEqq2mbQDt27UJHiP4JK9CbjV6LoCJbeFpB4KMZIaYoVJakxbW0wIAs7PcotPW3T25TeYyOcHucHFRI1+5j4AAZSlWRmjgLgHE4OiDLVkdvAHomV96xs8WbBrkiAxGvhVme9ourwB+5MV1wza1BAHqLsaLZLlx+1GCQ2LccXqSdRpZVoT+urFUZTIilugagkeQgaSskB7LIe4VGuJTLLTetSmCkH5Nkeqcpxxq6CK4LqGtoroKreKRlgoi24NZvLE0ABpbTXUGsQhpMst4Joip2A4npfNKfpDnDDYxYLjmphEaq36+Ge/E/r+jaoIjkabuV9xUnKntok8w89IWa6diLiCP/FlymU/sFPQB+Cx7D3X/JhChIlNcWENOck2sQhlxUUydaS0o3dGdXYtmJhPTG6YRyO9b4f9ptiSeZqATzz0VhXxnNwIMVpyZKIoYIEvKQoSR/4cej88/z4cGo3khDwxoZpMSEGVIbj8Y9Hc4C+0icEAJRyIGS009vLhB7V/vc80nQeyR5cYmQTG06BN9hP7iT7MpNrfZFmMbASSwp9H7BeYTumHEUi2YP92kGOnJUGGTt4SAsH3FJ8A02ALdgR8MXdLLz7SuOCBFo6LEwH4WrRQL+EIaIdgrNwMu++jS8scmqP/AmbaHXiNIRGKhwK+FikUxBVEZxqIiS/zJFt7fESPj9iLV4dF9VugtMG8BUSi5GjeFgFy8ZRA9OT+V1bhzl1l17GQ75Z2h2Cb2ww3No/UkQ29TDAGKVRTHwynlQ0CNzH1w93ahdSwaxR79/3DmsUZ17bju5Dab8trhW0B65jLfli2qwIKDv09Qwu2XRllUTrICxVJKSkLIqlMlF2tsXbaLCJjJUfxBP3ARsE1oaimTP8YOTNlQCk0FUV+IwSR7LYN9Q/h57KEhZ+xMF2sE34Dvoq+llViotcELgkBQoEqiGLHqMZGh9xfeHo0fvYNOKRZSkRbYuaC1MPTIUBSiU05kBM+U8pCa10ZQWsSEHwYLAnAAU3LNaUmJjgZbkOKQABZrl8XYtPOoP/ZgP3w5bOCC78J5oPvhuK4eu8K+TWYdcHIccwzvHyc/hiXCDRkvzI+K97cnetgT0hPCLjDsffnKQHDf4/Gj/IlzHw077ve1qgR4P/mjrECwWLO4WisuQktDpxU6P2ZI7/L3X2L+eDPqsuDJzUa0YXUdnKbZDmutRhiAeEBlufCMeGBFpq0/AOBli5yoH1tpGkDbKMtwc4YHo6jUojyPwyVQTPpXXNTpYZsTIPRMRC6w/7mkHtIt8tHNAQXGjMTTjNtEAgizPDPPEo2EMJMsTdvgW/RCMdwIksDt5dpQt9Pedzb4zTt3lWAB35GfL740d/LrAjwWaKmqUgEAzISApMebBErm6QGGjCHHVhwKeEPJbFmPK+YgU0Whg6ZapaTWdy9uTBFM5c76iM8jXlU+AYEL7Gw7N5mY1jEazCTawoykdNhRY96aCBjoTjaj1MawiKLY3XQMoITkRVbZrOJwEsuOvGyBWmGY5j0aCGNYT8XMXvO06kzabWUYG58bFlPCrjdwW5MU38CCUZT9LosKlHo5NHdzMEDXD3MITlVRtIxixurYpNYJHM5HmhMccVhuuLcHbNiuFawl0fdi0bIXVAThRTsvAFpJmxgMJ8hesxhoqaMaWGhqgYU6NRX6lETuKnuJ44ASaoG8wq4xKjTOQ3wLVBgqF9BC3ihHWWYO9TZBH9l9ySWh94l+Nrip5bWDoO2mDNbcQxgvyAvIRFDUg6I0I8eVxSyBbxzkEdySZ4O1smnYDzkrQFzKJ8KFEyaYRiGpv0P+Ts0Njw3EavH4PChv4APBlBE4AOET4/4JeRpK3qBAAXsiBYHs4SEQmhqVr7QbjoM2w2FUOFjk9jjPdesulaCif50sIWxnICIMwoGlAmISdWpeDcbHc5nh3ObwNYnGAlJbL3jdNVcAXKtC4Jco5nGby6NNlHDNaXZ90wEfFsGFVipGNy0Z3V/+XM2oc5zTITNrGqDuQ/vVDNkp7kpVuSXIfGiGMpMjCoXgKz4mGp2UFxwCaAeW3kfqfBlfzfFJkSCfokGCZ8jFKu9gaEJOPkhRCZRErdqhH1OWc8RhW9E4RxE+dcJmrBZShmPiKrenIyPi9dMFhb5cyEHZb3OXDKvPckIbVs08KloS0Ke8VzgPKTPu4zGUzUr7o9zHwldI4kSlPpBlANzBdTsQuCj0xeG62PjZoXFZjJGO38cOmIc7vykJatFIAh3eL/XRqEWI44pqOcgNsSiwO26qKda2QxFsRMurhSzrRgfymI9IRT5/jz8uvmoKJ87KcyEtn1QETtSKjHK6iKBWFFS3ENOFrZpq0G6xSJTLI2Tzb6a4GYuSIHzxBDcK2u5fKD1foXigBY6SY0h4AB9uKGFLCe0+T0M7RSv0O5xuVpBlgUDKEmVJYVQ998ZGd0sNRS/6GaaArj6X2sewForAXAKvozHs5mQecnBZiKYI5IEFyKG73E8uftTJytGYhS3ksBISwAZGJR4cZ1WY6I0EI1359Rft+ptxc5NDMQgKrzKPkE0WdBIhpJlKZQynxFPQY/fxqQREF2zRr3+RtMGKzug7d4w6VlBYljNkS8UTdugjUcFvACvyzYa417ACaE7Tu/E849cRMiV3B+zPbWRgHUDkQ8MxBwN0143SqhA/k/oDrSXxYy49ym/8a3FvVBzLgpJbgRM0AZAqSb7PRp1hHEg8JdxKZeaCpbiNteHkB/EptdW/YolWBNeB2rLOsoZNbbam++E5QZeRc3T0LAin0BkAS23K6g3AL6FwJYEOuTWH+jmAiv7r7foANV6axrFn+iLd9RUklWJ8XlabzEjWEBMAXR3qLeLwPdgRXQPUVcKTp1ZaKlAS4CWkDXS4ppYBabgNc4xBwzVF2R504MRitjREZLhlezni3CB6N8NwOqMGk10Nim4vyeoBJ4vmjgBmxYoyCoE7ZdPzImGrntsrWYpy3mtVlTyfhiNdRxCZvg5X4pyY1GIw2uOOzQQvjctzn5b1Iplmx92FS6JMhCokbNAUYwv3hts5EU2heE5c0LQZ2DCXbXFSqliJXNsF19zXLwvVNMHcPAfy7vm9W38b2ecUwaS7XyJSaFHZF1WEowV3eORed89uwMB7hRNcqhn2mEX4QEcTx3ASZaokMQ5GCdvoOF1Y9OR9gultqYw1C40usbh0Vw4xAPV6zrNB4dyNHzLzeBLFZMhrTwOy9EwzhbwUwVzba3zVNmywZSqBaiVi6KvOgfV6mLmyPXrXbN/rQ8PUYevcAbcDTU0SzN2lRC6m3Bo79fQ2dwQwstQGe/2Tl33QyxCqFIJEgIvpnXJqaq1rNIo5bm06/tEXNssoxIRbKP7dhF3vBYhbLtd+yWOrY6MSUvdasGzeXIv9n6R5YK7Sw/NqWwXuSgi7Py+TsBzwONdWE0DHCjOC7kFh+TfqFi7ovCICrpwSWZjLLWjOlhYH9ISTuPQKBJEcLstdpeXRYAgU94Wrw/EX9J8UlkVeTBBsEtfGbNp7KmchLMQFCTeIRQVDAxi9ACRa8Zh6TNUMYaYAP8eIH2S0A7GW106F3tfpz4OcPrNrYottZM/G5EtdHXIuUTFubHJLj8O+UfDcaojerO414km1drn1eK86ik0WNKUO+1kvyuVvqFEQJR/g0qryiWP+u9JsQi/W1Go7TdymzruT4Y6sVHBRZxj5Xoa0V5+anbKwwm6dWjkGXgMgns/vIWZMLBcN9FWFK47Mnuvs2NsWEJYR0XIT1TOWrefJt3yDkTbYtPA2/S9obPNncTeYjCt0KIMDrDMxpOxAaZ36Z5MHxddcQg+laPapjMHMZUnhq2PytucOpSfCVhzp70x86n50JVI1YFiHkub9ueG1HFmE6ZBcOEXAogOD1SO02rPgyMiqC6b2962eDnb76zNIcDQaZ7vTt1IrorFTlHs5E3l0rTbyhlZreXLdnONkjpCKZw97Ee/dnWSRKDW8sFIyWphJGnJgYQD0+tAbgEBA2yKogVBoZJW6rfbN5E2tj5uZEFsW13dYMJ2V0AG9URHvtJUyNZID27u1QKGJa14bst4R76L9ZIXZ0BT37fAI2l2yzL/BNvW2vQgB0Kcf4V4ZvYLHrYC3mTrTXPPI1Bnr2SEV55ZpfPbKM9xfsY8zUXkEbM2WYQINnGeGw4sl5JOmY017tB0dYemh6vVGq7Z3qoBxcQNHBrPrQgoiAf2aSiVEKJbQPkrHjLnuDAjtvdoPdSlB6JNuQlauzYMu1JqrV6t/Wv0k7IQY6tWp/5dHHZKQkP/tiCoLW10NZv2uQa0Ed056pV44koTZC/TGlMnggS14ffcViD8DJBdAb7DW9rrNe3wnvZ4Ub3e1E6vSn+UrE/5kq4WrKgysKXByoUVTWp1nspqtIYdftM94WvYJRi2U2Z/RyuImLedM1sNC4EmqM4XRn18MJdSHra1un7umDpF2cRmpx3pT8TObDc24hYtvQ/dlJHQMYy192Qp58Q64CsTMV0D9+6nCefidp/vvlMR6DY81wZT9lrrsJ87g5YLIObhvu7i6l2XMyhIzstdcPRRxX60jWP8uxrZZ/bbJw+/6pji7tBBf/M+P7K/h3mJA2wuKE6tHjPaQ4eFKzWaVSXohIIfe2habCdRqa/7Q0R9BYdf1mXXXSV8NanIU+ISPshbRyiIYtaeu3BdruABP+cSESrdMw6RG3gOFQFCazC3cMMOiSmUsFiFJxV8vM3FPPMH8hLV57ZZIneL+32iorwL5BU/EXwLIdFe0vFWLDUyo08VbAdGp1t7LnkxtkF/PGyiA7Hy1XxvjKzdSX7p6Pv1kkH2dK+a0Z1bQd12Z+3suUK1q0JkOPj/EkdKaChaGU1nO08qz7/tLLW2QXFhjJpGMx4LpnUtD4UhKQwr1oym5vromMyiSfbvaieMGWw7s21t1zJHEHObSmUl9SuTfgHAsXWxYqKuNexxlHa4YhQ95cmhLqOMKMNjq5YP5LRW36Q0dekl3RBIQPHCWNOFcKHxBAp+tZqZMHf2V4uBRyPyZNM9B++f4E6DvSv00AK+hOstUMicfII9p1wySkD9SRR6wOFXiCU0bLRK1hnkpW5Yno5KjCyjHRsZwufBYACLdjG/l8iwnq4pMcY9T9Ey7O1N5W/dna3l6EwCi8YO3wkjCae+N2zi0Axv61tv18ByxNW3CKQrqtLHVcbCq6kLRuqZDVcoWs0j1ofgWoPYWIJGxpp0dccT3EnFj8/suJOqZ8mcCcSmldGiIDqMeJ/BBI72l4m4pibscpSMyxKg/Ok92FqsbjtOXL48IoRl1kQbMTLbRwujFG7+AOThFLciEpThjaPxIdTdRuPn+Pc5FC7KuqOwdYGDSRrXcO5gKBmGjHk5FOdcw7LsYXXeWYQMRWWX2a0vwupCgMqxmFeOtezd7jVWbFErF0Z5yS1dgF9AGtB+eYmCKLmG07FUqsWwhBvudmSAGAStuciXssO6bQw0VAZ2Dos6MoYus9Ga2AzHhZ64U5GHfqRH3Cns1IqGbslF2xv/tuFth/5bkOBXd7S56bG+zl4/55E+zqP9m6/wbb7Or/lWn0ZZXJCdaPg9b12TtDYzPwnSJjrWMk9ML4bKml19vtN24numy5oxwLbag5kGOYuG1c7waDiCZEID2rLkHqwks+kf8dukeGBBNTfulpKNuy+YwgAIFF8eta6a6r03wbozQR7aMd0rCzv9KIue8LrVGO5dhR2C1TzyBtboXB3RFLdJwcsKnDDV4Li63mJc6oK+BFzcb5BhpnGclku4UdToGSVpisNQl8AfjdDSGZHRiOVyeHmEKB7iRwCn7Xszd4KjYlgy0XdCU5dn7sdN3wOzFz15qeZOoEIOjMwbFnaCNm7c3AkYdhMArXYD0zdu7p45cc4ImV3Co2NREtzReGd37hxZ6+B0x2tZjw77gWghMRqBTTdS6nLUsgUVeKNm1p50f5Wbc+cY4WFuBbE70HEInLvssAFdBWO01rGX3v1GNyf0FRja1YVThBrpZz1B26nm7dS7oQ1KDmxAU7JHU8wpN+CNOxxz3UVHf3jjkj8Yo2vXV4zd5QsbfjRv1elYd5n1vHm/wS8TRVnRLYH3RqYJXX/CUzI8IdQZNBTygTe0L2joisVztaUPANnBgPnjos82iM4G866YtO4jLB6n1YOT8G/rJjsXIPWUE9DEjYN3Tseot+GubywbimPcRnEsyjD5nhr8H4OBKxiFXQAA", "sha256": "02544f0e16f4dbebb0cbb6cbfda3504746e5d90d3fd485eb8154d0bec1367687"}, "validate_cfpb_v052_pipeline_smoke_v02.py": {"data": "H4sIAAAAAAAC/+1cW3fbRpJ+56/AYM5MAJukLk6yGe5oHUWWs8raslaSM3tC8eBAYJNCDAIcXCQrov77ftU3dAMgJCfztGf1IAmN6urq6urqujVc1/2wWCRxypzbMInnYZnlzu3uvrPA36O3Zz84F4zN0fLNeN9Zx2tGoKNilX1izjwOk2xZsWI8GJyHd846z27jOcudrCrXVenEhROvVlUZXids7DiXN2ioB7nL45IVTpk5YeqwzyXL0zAZ5ExCxFmKAXIWAfZ+6FQFYM/u52FaxhEnbslSlocliCuiG7YKiyEQzZ0yZ2FZDAq0cNBfq/lyxdKycMICw6yTOIpLhxGhacQcYLgBxeUNEZFG2TxOl3hiKydOnes8C+cgaVklYU6dc1YUIKzAbI5vWX6vZppjtDgtwIH4NozuR1WKt/EiBnGgaXANzDerMP80AvOSeBkTQwau6w4GizxbOUGwqMoqZ0EAhq2zvESvNCs5E4rBQLXly3WYF0w934TFTRJfq8dfiyxV/2eF+i9nYogoSxLwkhCqMY6yKgXXxXtwnJXxiqmX6nno0O/fslTiWYNfGFSBneFRvCjv18Q62X6YYs3eYYFzrKnsqRZPgvwQFux9NmfJEJSki3j5Jo7KofM2Zsl86PyspeA4z7N86CyoPdDiMxiU+f1k4OCHYy+iPF6XxVhKBfqNU7bKAgCHwZwV8RLt42ixvg4gy/uBEuWAizLa9oMoW60gdJI8j+Omnw95GCXsPLsb6qYLKV0/SeGq39AqBEW4YHVTAiEKMo4kiIrbxgtIeJbPi0arEt9Ai28NUNyE+998GyzixBiE76aARk9Eo/+lvIFwQ7xAIvGowKbfxqg2h94cvz38+O4yeH9yGvx4/uHj6ZuT0x+Dyw//dXx6UZO4zCFwtL+CBbYoxL1QlLLPEVuXDqShSthpVr4lQL7uE8f5s/OGqwHsPxZVXC8IgSN1Iqb2VVGrinE97y9ZbRoHIswmDriR5WyaZqOczdli9n9HDr58hZ/BlT+69oNBcHb+4afjo8vg/MOHy+D49GfnAApszNLbOIeoLlnpuW9PgPfi7OPl8fHPh++sHq4/MB/RWQgmqSavhdsfY+wsuWWeYEy8cFow/AVLCiZxcBYHgdF1THzEYkxfzQb+4OI9ZqsGt2jZcVxxQhQ79ebbEXzGCt3vbFuKDOdHjiPKHZx/PFWojXGAOK/SYH93/9vdf9vfv9x79c2r3W9/ATjx5+TN4eXJh1ObH43exjlb7Gi1Squ/U2RVHrGgewB/oFb8/PAfQK7pI6Th3Y6lVPhTwcod8OufFStHxMli5zoso5tgl37G8o2r0Z6cYpkbiKXkzoM45czk6murBO/vjvkGaOAM3h+enrw9vngKeQMbbw2wDeMFK0qOuUZMQvnh4oS4XQueIQG8AWMoPkQVN1l2jPPdloEgrOYxuCH71Wuw9+py7+tvvt5/9cuOtDKCf4Y7iypJBA+wcLqXnoGCnMfFOitiGg+Ae3IS9VJKmTl+Q6xpSRAwSgEB5bXR1+Dx+THNejuKnP0KDd2P4eeT439s738bs7u+3mcfzi+fmAAxAMsN5UYMk3wY/Bm2KcN2h3FDZihMxRyGEIyjlDqESXLvpCGOoztnzgCwitO44JYMQJZQuPcOLNDoE5mFp8fnQEeGqLZAIywy4GED3jvXLMnIwMycMyiTeB5nO8U6PIIhmTuhE4HIa2HTkr6HjXh2chKcHV5eHp+fXmBqD3yFXbI2E3cC426MY2yNPeXl7tX19HD0y+7ob+PgLy9Hs5ffq0f8fzWmh9nD/vDx6todUscTX2hgd30D485GptV27nqv//6nq7nvvZ5cvXy9Nx2Nr4rZa/81PXuvr+YPrx6v/NeqmT/LB/z/9aP3mjoLuVTjFUXaJF0OwruP8Huf/9bdVc8q75j0TVmui9eTnZ2ri5ebq+u7u7urMf5tTJK4jl2wjEtSa9vH/9vQGPRxcIFDDJv7Z+jN4//+KFRHk09EBNghnZ9NwdL5BkdzzjYlSxJnxTaMzOxNREZuvvKvrscPu8NX3zy6VvewKO5gAGxgRW+i21vgga7gogUbeYMlmo6cGbfQeQP2AABp+zthFJElv1H4eGMUAleRRdgtjkK1Aet9Wn8BCPYMoQPenpxfXAZnx+cX2C6HR1KVbZnlyeaOkTB8lSQb5y6m31GYbuCL3LKNE66cZcYdqMy/Kl4oggBPen9TVNeruNxka5ZuYnhI2ENLCPsGdliMs2ezyBn7jW2ukyz6tAHaiCUb7HoGnwd/FzAg9Bwj6GugYgW2DKGQJzMxuQyjsj3L4/85e3dydAI1cfLjf15eBEfvDk/e2/OkKd5nlQNy+XRyVsKp88kbdPJ4eVNiWvXuGbw5OSctD23//uTieDvLJNc2QHNvsc7fYLSchpOTE5PyYdhlVTKXgvJ1Q1AsWIM9av6rdQIdtYG6io3nuWQgUz1ZzUuJY66QzNvcO/zh4sM7GF/ds5WULaswh8ZjbIO9sIpBFQxFqEpsg/tNxHLiZnLvGywcXFyefzy6/HgOk+78+BACSEoOtupvLMVpKdj4oLWRi9UlUoUaF14/jmeu2t3hFrCEpcvyJgA9K7I6TLgVeZ+BjiMESQj1HHBH0wSr0k9pdpfCdCcsBRmx/LTWgz4OyIyNEmxh5z0iBOGSedq79YWTKsbiOmCJOdYer4fQRx4euAhpXAOjsErzLIHNLf3nqYvgRw6muRgABwl47Ar7m6QdymXiFGUOpNx19nA6yUkf7NV0vZHH5h8kzGTtxElAzVTOWFBU3KeQc3JYIowTFNUKYY/7PgKplzJTYBOxQqJFj9mAv/2+4fp71gJLwr7n01yx8iab8wbIngo2scDs4EUJQkUrQXXRmITvjP7DbploQYCrALI91dN3QC5FiR6+HjrfDp3vHmtQvoYhtgDFMSrGXVmbbGdVFSVfQOwKR2CADfCdJkzOiztveBFj737WZNOwLK1WXHBrimwCEK/iBhcYLySIZsDxOH9x9p2DA2dXeDmGXFkIAC5Rj0kgnT8daJz2SM+YLiEoxKTDhMf6Sia8Uk5aPVlo3SpPNRsgwLSSPPAVQCaqiLzHuXdLA00ozsSXbA6BJZEZUotcMw6CyWtXXHTSfl+MrUyzjuSLIcdiMFGSwl9u74RRjT46KqV+OOVzScaYnPrCpEMsFA9/cICfoATfMDriRfzDFilB0MNjk0Q5iE2faJSzEiste9Z4BHfJ8AjSZR6uCq/Etuf7dehgh8PCBenfcR5DI0/LCseJYPR4PJ5JTlP/QpwJUPhzGMw4E6bh6DeYn1/NyBYjrJj8Hcs937doqDlHqD2OaiqkdCKl9aWTzjp2A20CHDhQtqvws7c75LuT9we5Topue3IsNU+tD2DKyzM6u5s0hEfodK7hr++h5M33NqQGbcRoJq24kLNxTmHCgUX0Bycq8VPwUiu7JnYuWZLBCBWfC34hug0xgQLTkfURIu446ueYS1XyP3ciEs6fxzzKLBjOexrqFeRMZ/VsY5JStHvAIKIu+pgDAZKV1ERgmj0cUgLqrcWhEJ6j2Rr7SRB4wCdKo5i7nhNHFPWdt7Om4Gt2CNzD2s2UsyZpQey7WqUkLh6cfQQZo5KmBHM0MB4xVMVUu3jwLeVfoyFPkLt4BrPEW8Ep0pT0jshuvWxuaT6BcbiGVTz3FoLn5EywlHZxEN2QiM8nDwLHI1SlWAitC8G0lnrUS6jcZbV+wHwTX8fa3Cm0Vyn8JG4CGVwRDQ1WiUbNsBpGNtX4vojdypqS/G6T+lfSQV49TUtXW3zste0EJyxNrdgEbihLaSxQKJ1hDjswtHYjUbGdoj7j1e8ihDbPoE2qOoID7vUBDjq6Xu8O21ctvSTYu0RAVyZV6uPa7xlgtKeWRFMn9zaZLKR1VfvYHN43DQaB7JncadrsUuTr1aTjhMwaYa9o86XR3iCYzDVbIbURXqXu+Fe4sJ61SafKDJIGN9cqhinWOf2ZheKlM9VQ3UZyE16DW9axBqpFpj397lk0JmG9e86Eem1DWJNdZqR9bKchJRRllI3GMcNbY+x02B8N/Uh6QMCPCxbm0Y1nL5rfYYk2VSoSuOwzgsTx5IEoeDS2G9DDaJGYyY+llPfVNcURCjKvYc6QFr4aL7NbKCt7bOnG9ip0l0d34XyznPLN9tDyWFOGEXHEHkG96R9CosEWWyPjbA/hTonHR4dIn1CAd+a2B+lHDrZBTJG74YbTOgkjdpMlsDk6h2sFzBRvbSF9YkI1MKJpFLa+xTlIeSfYVGECbiJjUJT2yB1hrD84dpWGt4izko0VhDx1HsDJjFf2uJ2Bpd81csKWmBysHh5sKroGs2NOW0YhpdwM2PxBVsC4QrCHEXFzFs5lboEHebQ9IpJGwouAEjJ9CssWcutMWFCnBelwytelNDXFpCkJloRrMjct5H+1kDf0gdL8svP2E0fi/I6w1LIsjEaZo8TI7cTl750NaFIopm4NmqKjuatK5JzSwp05f+9NrW6fF/y/arFAlQt2S02SmhrXOg1/pftwVHipNAd4W73GEsAf2AZ8geQKI3hYJRJCjo2g+DxYYEuBBwC8zrLEs5BsI46bGmRwE5UmloZvQWefSBO5Mof71HStTE09tTmLYir04ceaxPicsawZYhvWB/C/gArya2QqpZMW5feo7HUHkE7jKZ+w2yMTzVOy3ml3wAgDWcIGd2eWwyCcksl2b8c6/3+/I2Z4BoK4cbXmVrnG/2CpMreRxKVkO7I+l3ll1EooD+WWczlMtgEI80tVdAHqbQj2NsB0jVegKry2AK6ra9SgCY3xBOgqvA948iggOyQJZBZ5CzSSvGUeX1fipJKpxP5RHo0SEb3ojXKQOnFKKWZ7+V3eidJp9Nfw++S25ok24aHX79oqFWB2WYhIFJoKWqpz4UYAnrwO2eabfZqbKKCaPca79G7BPhzoa1vS7a0qfMV5tVp79O+ByzPLftOU7SXBDt5SuM5q9S23+imZfFIerUhgK4zC/8qoGVWKVGT8hUsqcixlDZMn4wmMdAezwmQUKaQAkwyQCXgTQJdSSQiNJeADFxYyilR1xc+6Yr6Idr3HXEj9ijmNEmS3EpFOd9Yw6LN8RRY9ws6UqCyi+FNcAiiEg6FDZZa/LUroPgkQJBjyONIllEi9oqADC1GgeItcm1yqbCrnQhg0WPAXcPHWBN+Nk+pP8ixiPNSlML+vkjJ+Ryr3B2z9PP4NhZqGE3/CobjnLupaI14pt87D5SqcQKSgayEciIPKMioStlEBXxzyEg3soP05thQSyyJsb+uUI5k/HIk1VLLAI/qwwmMykkwe/rvDw89IQFPWlErJJBivXRsbjqHgAQgfmBFIw4AQI8q0EyV3mPHSOFdLlGNVKSV2XnCjQ/SbSoSzsY5AUNWfMjf4caSinjiP1PDGgbVp4NYyakY8hwLoKXzavKG9LifTtLQCJVsHzbNsFUd5FmjZwj7eG+8Ou2CE1PUALPaaLx877AVeCM0oY9chhp6k/0D+9duBIxgMefwZ3SUiZAXgcCBWX9D2a8RWnrNe29g76wg01LqkRcY2EqRQ9a6w/4U0iLWiEAEtCeqX94ZOQKHafhXhdabw5FSGrckh0o9tjijMgVhgmE1YIpT2xsKoOdjtoO7L5W2BvFXp6Ra/X/gEtHj0t4uhAFvs+S1h/Ax3m/Z6ZSxU//Y7OHCevfctk7RjQcWORdEY2Rh6I8vjssCGaGaMmibRXSENFN3ZLE4QuwYQ8r/63YsX9soYvThLEA0pA3XkAAFvbAXN7ZO027erUlXrQRmXhm3DaW5FB7cxyrButnJfOzIqY+5iFkz4x9KvqVOaMzPgURP6RHK9FQVcuG/UqErQ5VHkiMF3xMiSasrpwOx4qAd8tJPhHWpOTatoMZFC8k9zw3Dr/C/i96xD2fUT0xQLK2tnEGLF876MFEVAYCi/ua3rjAalDu2F+yLl2L0WwxZMB4/aQFqNXtM5d++2IRp6dbhFOkw1S8pi6hpMIBYbipU0h9CCbeY9jUfq2w4kUvU+iQFquKP3Yq+np1Y/dUeLEbZF7HWsT9d6NNmn3RIavZm8R7BESLEY+YUsVQvvAlFrP+Fl+9qxMGq7zTd2Wbf5xiyWputO5jsdQGm9Ufun4wUFiTqaeRVys7l94cMAabpBtXPVA7TVV1pniEPc25UpjWJ3nEcirOqpu3EHblUuRt+5dVBToBHB0HWVg3WI11Dqz+2/ZyJCMoPt9UMnRIujaFEHWdh9F1GBSfVFausTwx00XQQVm7UmzwoP9YaG/hUuOAfiDilS7nGS4XBPKOHTwPXYzow12c7/x2n68OjzFkzd316u1WL1oslrKRoqAYtzERhV3kwPPpWXW9zZ1BWXkaAVqPShvpjk2RvQ71vvMwkqtia/3ugYKeCueffdc5AU+U2Kmtu7nyY5gipRNJVDJ4Xme2tnNYd91t4igfeM2gCNojH/WooDKiSnU5z2C4lsnSZGjLmFASte8Yya0FEiPqK6cxns758zuP3IYUfk/fN6FXP8ZyDo2CONrk/ITHtZ5hkTqmJNVzXKWlMQfxzOKpUNoUPDsITrM6R9S6y+PlW/U/asZ+IZOi/MR3VVrHcadHtaXiXmJZLiPrFDBds5j6RwqsUEkPv6hDsjOpdMCpaMPvMWYXPbqck2weoJ+42qN/JpZHnTVNeBzfwJr30i7Up/pUFYaO9DOz0cB996qqn4kr0v62JxrbNa82A500Yor51U1bFEvwgaTXsK10x60WNmU1t6Eouv6VUNDjd+awCuSpThLKb4zFXlBXE8JEcVFvN4gXqAwrqVKaeuJLO+tm7ayFvuhHpbrIZGsUEnTGcE2iwWFUhUUZ5Jl5Hp081Uetlgkd5CLRw9zFu4F+0b/KpcWiISXD15I/y2Bm5dKCdC2lLMrcgyL4RslHQ2nPk6YTZBraRZXqifRXJQVVI+GtlQkazAkPKmvSq6lI+e/4wIOpHz+LwYuYa0xX0yaLj2dZWnubObed9GLgCdrOpZXmVaL/KwXiXLtfQbxZ9qGab8caaS1uJlK0Il+KdSfa34jeaIdrIdsT+3Q0remR1kqkMxjvbEkGxHrtg1Bs+2/IemaBjhHc9yA0x5sYG0S1CLkAQwNIpxl9sTsA0mgsqZlHLpBtQxPb2fdWCmtd3rkWQM5KB5R90zeg5V/CRQl4oOyL5o6BgV9+jXld36UiUbFDnP1Jf2JAwW9CauWqIxlAiGLWk5aDYY3qrhyz0RFpS3TjEPEVN1KZ067FA1AYpt8F59gWMMveapj3CM8c4fx0XG3QaEPk2PRDpfQJ2JT7oE5i3vIFv03/x327lXun6sC83bPhCdiyQalOY1TR5YhSEv2iozr8tu8schDh/Ya5+tCRAOabVPLJPdRG6nQU0rp7tvwxIyetu+bndvG8Ya+xm+x6Tf8WhM3Qgc49HvFA0TxtYFhj6aWZiVQurpqpVUoyfXUr39uObqGE8EmVQNAK8ukdaCrdqlZ+v35/ANEmpDo6uH0hhaRbV6Uayzg8O27kAvu8GEFJMHyMOjKQ7CBZ/8fx3K76lDeWYkhBtYBtOpIrpVnOGirKDABVvcBKwoXNT6EhS/3S8rn3ARXwgxbEpcUMPN/8YVf7dBo6ztChNy3+9l1K/gX4waO0eqTladYFA+2PoJP3Tpli1ddptTwh53CRACcJvFHbX9pkulbxp2iLHPh41wpD90vHovD+2QpHzJN+zQDEr6LS+bn1RTLemzKREza+cJpeqX0Qz1aZTfp/ybqbOGquEUNMG79b2tWR+bwVbpj49Xn/DJIE8659ySoWBhTFbCJ8OwMXsKe4yHbKzv/Iyp7gfONAcFlpTqT4IQdRHxgZBefmEtLQ/2h9xpCmC4iTENSptRIPPzPToezqtIzCuR+CgY9CeFeNXnwcan4BUPEMgwLzVSLl8DHObLijbCGX+DwL/4ihKlNYJgnkX41I3RcxzO5zQM7+K5oxFU50gG/ob860AHPHxNN25DhOsPjO/D9OJRJ/NIGHK9uPgHXHqxcSQjHQZ+Gpv+HMwTRIovuhmHdz9u44swvYjNbdyDTn+ZpX9J6k3ftyDyCy1PoJIqohcRfajlCTQkqk+hoS+29KJRynukT3ATZW9XoYLb4OaV2Xxs7iK5sSj0JrZU7SQRgL5nJ6Fr9WA6x43slJ2aOqCu4/p5aHqohpkq4Oy2GtY2SgWs3TbsihlzJaYwt2zWuot9qIgOum1oHBPG6SInJptMIH3OKBBqGDYOGwuAa9HthdOFAdz3eTTDdRXA0r0z1CrYAFn5oyqc5IY+IxbQIYVPOFI2PQhIioJAJriESA3+F5H6FH1tUwAA", "sha256": "2e9b9f47f51bc3571366883311f1f9200f839335c06aaa7c6e16bf7c602c24de"}}')
SNAPSHOT = RUN_ROOT / "source_snapshot"
SNAPSHOT.mkdir(exist_ok=True)
for name, item in EMBEDDED.items():
    raw = gzip.decompress(base64.b64decode(item["data"]))
    if hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise ValueError("Embedded module hash mismatch: " + name)
    target = SNAPSHOT / name
    if target.exists() and target.read_bytes() != raw:
        raise ValueError("Run has a different source snapshot; choose a new run")
    if not target.exists():
        target.write_bytes(raw)
sys.path.insert(0, str(SNAPSHOT))
for name in EMBEDDED:
    sys.modules.pop(Path(name).stem, None)
    sys.modules.pop("scripts.generation.nemo_data_designer." + Path(name).stem, None)
# The existing validator supports both repo and flat imports. Bind its two
# dependencies to this exact snapshot even when a full repo is on sys.path.
for module_name in ("prepare_cfpb_seed_v052_pipeline_smoke_v02", "cfpb_v052_pipeline_smoke_v02_common"):
    loaded = importlib.import_module(module_name)
    sys.modules["scripts.generation.nemo_data_designer." + module_name] = loaded
import cfpb_v052_nemotron_exploration_v01 as workflow
import generate_multi_turn_dialogues_v052_pipeline_smoke_v02 as generation
workflow.verify_inventory(PROJECT_ROOT, INPUT_INVENTORY)
workflow.save_json(RUN_ROOT / "input_inventory.json", INPUT_INVENTORY)
workflow.save_json(RUN_ROOT / "environment_versions.json", VERSIONS)
MODULE_HASHES = {name: item["sha256"] for name, item in EMBEDDED.items()}
print({"inputs_verified": len(INPUT_INVENTORY), "modules_verified": len(MODULE_HASHES)})


## 3. 準備 20 個新 seed ID，排除舊開發集


In [ ]:
import pandas as pd
RANDOM_SEED = 20260919
PREPARED, INPUT_MANIFEST, prepared_info = workflow.prepare_run(
    PROJECT_ROOT, RUN_ROOT, INPUT_INVENTORY, random_seed=RANDOM_SEED)
print({"selected_rows": prepared_info["selected_rows"],
       "prior_overlap": prepared_info["prior_sample_exclusion"]["selected_overlap"],
       "offset_invariant": prepared_info["offset_invariant"],
       "grounding_gate": prepared_info["grounding_gate"],
       "policy": workflow.POLICY})
display(pd.DataFrame(workflow.load_records(PREPARED))[["seed_id", "product", "issue", "release_split"]])


## 4. 固定 Super 設定與失敗 audit（無 API）


In [ ]:
import data_designer.config as dd
MODEL = "nvidia/nemotron-3-super-120b-a12b"
MODEL_ALIAS = "nemotron-super-exploration-v02"
# Official Super sampling; explicit non-thinking and bounded output.
INFERENCE = {"temperature": 1.0, "top_p": 0.95, "max_parallel_requests": 2,
             "timeout": 120, "max_tokens": 4096,
             "extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}
builder = generation.build_config(str(PREPARED), MODEL_ALIAS)
builder.add_model_config(dd.ModelConfig(alias=MODEL_ALIAS, model=MODEL,
                         provider="nvidia", inference_parameters=INFERENCE))
for existing_model in list(builder.model_configs):
    if existing_model.alias != MODEL_ALIAS:
        builder.delete_model_config(existing_model.alias)
# Pass only the explicitly pinned NVIDIA provider; never delete global cached settings.
PROVIDER = dd.ModelProvider(name="nvidia", provider_type="openai",
                           endpoint="https://integrate.api.nvidia.com/v1", api_key="NVIDIA_API_KEY")
workflow.save_json(RUN_ROOT / "builder_config.json", builder.build().model_dump(mode="json"))
SPEC = {"purpose": "exploratory_generation_quality_not_heldout_benchmark",
        "model": MODEL, "inference_parameters": INFERENCE, "preview_records": 0,
        "provider": PROVIDER.model_dump(mode="json"), "random_seed": RANDOM_SEED,
        "prepared_sha256": workflow.sha256_file(PREPARED),
        "input_manifest_sha256": workflow.sha256_file(INPUT_MANIFEST),
        "input_inventory": INPUT_INVENTORY, "module_hashes": MODULE_HASHES,
        "environment_versions": VERSIONS, "policy": workflow.POLICY,
        "sampling_note": "Seed selection is deterministic; mood/length/model output are not guaranteed deterministic."}
workflow.save_json(RUN_ROOT / "run_spec.json", SPEC)
print({"model": MODEL, "rows": 20, "preview": 0, "judge": "disabled", "api_calls_so_far": 0})
def save_failure_audit(exc, path, stage):
    # Follow __context__ too: SDK errors raised "from None" hide HTTP 410.
    chain, seen = [], set()
    while exc is not None and id(exc) not in seen:
        seen.add(id(exc))
        message = str(exc)
        for secret_name in ("NVIDIA_API_KEY", "OPENROUTER_API_KEY", "OPENAI_API_KEY"):
            secret = os.environ.get(secret_name, "")
            if secret:
                message = message.replace(secret, "[REDACTED_API_KEY]")
        message = re.sub(r"(?i)Bearer\s+[^\s'\"]+", "Bearer [REDACTED]", message)
        chain.append({"exception": type(exc).__name__, "status_code": getattr(exc, "status_code", None),
                      "message": message[:2000]})
        exc = exc.__cause__ or exc.__context__
    workflow.save_json(path, {"failed_utc": workflow.utc_now(), "stage": stage, "errors": chain,
                              "model": MODEL, "retry_performed_by_notebook": False})
    print(json.dumps(chain, ensure_ascii=False, indent=2))

PREFLIGHT_ROOT = RUN_ROOT / "preflight"
PREFLIGHT_REPORT = PREFLIGHT_ROOT / "preflight_report.json"
def require_passed_preflight():
    if not PREFLIGHT_REPORT.is_file():
        raise RuntimeError("Run step 5 first: no successful synthetic-only preflight")
    report = json.loads(PREFLIGHT_REPORT.read_text(encoding="utf-8"))
    if report.get("passed") is not True or report.get("run_spec_sha256") != workflow.sha256_file(RUN_ROOT / "run_spec.json"):
        raise ValueError("Preflight is failed or belongs to a different run configuration")
    response = PREFLIGHT_ROOT / "response.json"
    if workflow.sha256_file(response) != report.get("response_sha256"):
        raise ValueError("Preflight response changed")
    return report


## 5. 無申訴資料的 API 測試：改 RUN_PREFLIGHT=True 後執行


In [ ]:
RUN_PREFLIGHT = True  # Set True to test NVIDIA using invented text only.
if PREFLIGHT_REPORT.exists():
    print("Verified preflight cache:", require_passed_preflight())
else:
    if not RUN_PREFLIGHT:
        raise RuntimeError("Set RUN_PREFLIGHT=True to authorize a synthetic-only API test. No complaint data is submitted.")
    if (PREFLIGHT_ROOT / "started.json").exists():
        raise RuntimeError("A prior preflight attempt did not finish. Preserve its audit and use a NEW run ID after resolving the error.")
    from getpass import getpass
    from data_designer.interface import DataDesigner
    from validate_cfpb_v052_pipeline_smoke_v02 import Dialogue
    if not os.environ.get("NVIDIA_API_KEY", "").strip():
        os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY (not saved): ").strip()
    if not os.environ.get("NVIDIA_API_KEY"):
        raise RuntimeError("NVIDIA_API_KEY is required")
    probe = dd.DataDesignerConfigBuilder(model_configs=list(builder.model_configs))
    probe.add_column(dd.LLMStructuredColumnConfig(
        name="dialogue", model_alias=MODEL_ALIAS, output_format=generation.PipelineSmokeConversation,
        system_prompt="Write fictional English dialogue. Return the required structured output only.",
        prompt="Create exactly four messages alternating user, assistant, user, assistant. "
               "The fictional user sees an unfamiliar charge and wants to ask for clarification. "
               "The assistant asks a neutral question and suggests the official support channel. "
               "Do not add names, numbers, URLs, policies, promises, or legal claims. "
               "Include synthetic_case_summary and privacy_notes. This is invented test data."))
    workflow.save_json(PREFLIGHT_ROOT / "builder_config.json", probe.build().model_dump(mode="json"))
    workflow.save_json(PREFLIGHT_ROOT / "started.json", {"started_utc": workflow.utc_now(),
                       "run_spec_sha256": workflow.sha256_file(RUN_ROOT / "run_spec.json")})
    try:
        # Same SDK, provider, model, schema and inference parameters as generation.
        # preview includes the SDK Hello health check, then one invented dialogue.
        probe_engine = DataDesigner(artifact_path=PREFLIGHT_ROOT / "data_designer", model_providers=[PROVIDER])
        probe_result = probe_engine.preview(probe, num_records=1)
        if probe_result.dataset is None or len(probe_result.dataset) != 1:
            raise ValueError("Preflight did not return one structured row")
        response = Dialogue.model_validate(workflow.parse_structured(probe_result.dataset.iloc[0]["dialogue"]))
        if len(response.conversation) != 4:
            raise ValueError("Preflight dialogue did not contain exactly four messages")
        workflow.save_json(PREFLIGHT_ROOT / "response.json", response.model_dump(mode="json"))
        workflow.save_json(PREFLIGHT_REPORT, {"passed": True, "completed_utc": workflow.utc_now(),
            "run_spec_sha256": workflow.sha256_file(RUN_ROOT / "run_spec.json"),
            "response_sha256": workflow.sha256_file(PREFLIGHT_ROOT / "response.json"),
            "model": MODEL, "complaint_data_submitted": False,
            "note": "Connectivity/schema check only, not generation quality or privacy clearance."})
    except Exception as exc:
        save_failure_audit(exc, PREFLIGHT_ROOT / "failure.json", "synthetic_only_preflight")
        raise
    print(require_passed_preflight())


## 6. 預檢通過後，改 RUN_GENERATION=True 生成 20 筆


In [ ]:
RUN_GENERATION = True  # Change to True only when ready to submit this new smoke batch.
RAW = workflow.resolve_cached_raw(RUN_ROOT, PREPARED)
if RAW is None:
    if not RUN_GENERATION:
        raise RuntimeError("Preparation complete. Set RUN_GENERATION=True to authorize this batch; no model call was made.")
    require_passed_preflight()
    from getpass import getpass
    from data_designer.interface import DataDesigner
    if not os.environ.get("NVIDIA_API_KEY", "").strip():
        os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY (not saved): ").strip()
    if not os.environ.get("NVIDIA_API_KEY"):
        raise RuntimeError("NVIDIA_API_KEY is required")
    # Recheck immutable inputs immediately before submitting data.
    workflow.verify_inventory(PROJECT_ROOT, INPUT_INVENTORY)
    if workflow.sha256_file(PREPARED) != SPEC["prepared_sha256"]:
        raise ValueError("Prepared input changed after configuration")
    workflow.save_json(RUN_ROOT / "generation_started.json", {"started_utc": workflow.utc_now(),
                       "run_spec_sha256": workflow.sha256_file(RUN_ROOT / "run_spec.json")})
    try:
        designer = DataDesigner(artifact_path=RUN_ROOT / "raw/data_designer", model_providers=[PROVIDER])
        designer.validate(builder)
        result = designer.create(builder, num_records=20)
        RAW = generation.resolve_final_dataset_file(Path(result.artifact_storage.final_dataset_path))
        workflow.record_raw(RUN_ROOT, RAW, PREPARED)
    except Exception as exc:
        save_failure_audit(exc, RUN_ROOT / "generation_failure.json", "generation")
        raise
print({"raw": str(RAW), "raw_sha256": workflow.sha256_file(RAW), "rows": 20})


## 7. 產生 Excel、HTML 與技術報告（所有列仍待審）


In [ ]:
WORKBOOK = workflow.build_review_pack(RAW, PREPARED, RUN_ROOT / "review")
# Immutable evidence files only; the editable workbook is versioned by summary hashes instead.
files = [p for p in RUN_ROOT.rglob("*") if p.is_file()
         and "__pycache__" not in p.parts and p.suffix != ".pyc"
         and p.name != "evidence_manifest.json" and p.suffix != ".xlsx"
         and not p.name.startswith("summary_")]
workflow.save_json(RUN_ROOT / "evidence_manifest.json", {
    "files": {p.relative_to(RUN_ROOT).as_posix(): workflow.sha256_file(p) for p in sorted(files)},
    "policy": workflow.POLICY, "note": "Evidence inventory, NOT a frozen or approved judge."})
print({"excel": str(WORKBOOK), "html": str(RUN_ROOT / "review/review_readable.html"),
       "all_rows_pending_review": 20, "auto_accepted_rows": 0})


## 8. 初始／完成 Excel 評閱後的摘要（不呼叫 API）


In [ ]:
WORKBOOK = RUN_ROOT / "review/generation_review_20.xlsx"
summary = workflow.summarize_review(WORKBOOK)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Sync the entire run directory back locally:", RUN_ROOT)


## 恢復與交付

輸出：`outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override/exploratory_nemotron_super_v02/run_<UTC>/`。
成功後同步整個 run 與已執行 notebook 回本地；閱讀 `review/generation_review_20.xlsx` 和 `review_readable.html`。
摘要初次是 pending，不會自動判定合格。填完 Excel 後只重跑第 8 步。
重連時填原 Super run ID，執行 0–4 後可直接到 8；不需要重新呼叫 API。
preflight 或 generation 若失敗，保留 run、讀 failure.json；解決根因後用全新 run ID。
已完成的 raw／preflight 會驗證 hash 後重用。勿把 Nano run ID 當成 Super 執行結果。
舊閱讀指南仍適用欄位說明，但本版執行順序以本 notebook 的 0–8 為準。
